# Functional Programming II: Continuations

A continuation represents the computation that remains after a result becomes
available. By representing that remaining work as a function, we make control
flow explicit: it can be passed, composed, and invoked like other values.

The three lectures develop this idea in order:

- Part 1: what a continuation is, and how a closure represents it.
- Part 2: how direct-style control structures translate into CPS.
- Part 3: how one execution can explore alternatives and collect all solutions.

Run the notebook from the top. All executable examples use Python's standard
library; no additional files or services are needed. Optional extensions provide
further examples without changing the three-part teaching path.


### Part 1: Make the remaining work explicit

In `inc(inc(x))`, after the inner call finishes, someone must increment its result.
We will represent that remaining work as a function.


#### Direct style vs CPS: first contact

In *direct style* we write functions that compute a result and return it:


```python
def inc(x):
    return x + 1

def twice(x):
    return inc(inc(x))
```


In *continuation-passing style* (CPS), a function passes its result onward. Instead
of returning that result to be used by its caller,
it receives an extra argument `k` (the *continuation*) that represents
"what to do with the result". It calls `k(result)` instead of `return result`.


In [ ]:
def inc_direct(x):
    return x + 1

def inc_cps(x, k):
    # k is the continuation: what to do with x+1
    k(x + 1)


def twice_direct(x):
    return inc_direct(inc_direct(x))

def twice_cps(x, k):
    # first increment x, then increment again, then pass to k
    inc_cps(x, lambda v1: inc_cps(v1, k))


def show(v):
    print("result:", v)

print("direct:", twice_direct(10))
twice_cps(10, show)


For `twice_cps(10, show)`, the first increment produces 11. Its continuation
receives 11 and starts the second increment; that produces 12, which is passed
to show. The intermediate continuation captures k, the final recipient.

These functions deliver the answer to k. Their Python return value is None;
the printed 12 is the computed answer, not the return value of twice_cps.

Key points:

- A CPS function takes an extra argument `k`.
- It passes its result to `k(value)` rather than leaving the caller to decide
  what to do next. Python still returns normally after these calls finish.
- To *run* a CPS function, you pass a *final continuation* (like `show`).


#### See the closure before hiding it in a lambda

The same twice_cps can be written with a named local function:


In [ ]:
def twice_named(x, k):
    def after_first(value):
        inc_cps(value, k)
    inc_cps(x, after_first)

twice_named(10, show)


after_first accepts the first result and retains k from the surrounding call.
This is last week's closure mechanism, now used to carry the remaining work.
The lambda in twice_cps is simply the shorter spelling of this local function.

The continuation is explicit, but Python still makes ordinary calls. Nothing
here yet saves stack space or explores more than one outcome.


#### Sequencing: name the work after each call

Consider a simple sequence of computations:


```python
def h(x):
    y = f(x)
    z = g(y)
    return z + 1
```


A CPS translation has to:

1. Call `f` in CPS form.
2. In its continuation, call `g`.
3. In its continuation, add 1 and call the final continuation.


In [ ]:
def f_direct(x):
    return x + 1

def g_direct(y):
    return y * 2

def h_direct(x):
    y = f_direct(x)
    z = g_direct(y)
    return z + 1

def f_cps(x, k):
    k(x + 1)

def g_cps(y, k):
    k(y * 2)

def h_cps(x, k):
    # CPS translation of "y = f(x); z = g(y); return z + 1"
    f_cps(x, lambda y:
          g_cps(y, lambda z:
                k(z + 1)))

print("h_direct:", h_direct(10))
h_cps(10, lambda v: print("h_cps:", v))


The CPS shape is always:

- replace `return` with `k(...)`
- replace `y ` f(...)= with `f_cps(..., lambda y: ...)`
- nest lambdas to represent "what to do next".

This is the *general scheme*: for each step, we introduce a continuation
describing the remaining steps.


#### Trace the captured work

In `h_cps` above, the continuation `lambda y: g_cps(y, ...)` describes the work
after f finishes. The innermost continuation captures `k` so that, after adding
one, it can pass the answer to the original recipient.

Before running the following, predict the two results. The computation is the
same; only the recipient of its result changes.


In [ ]:
h_cps(10, lambda value: print(value))
h_cps(10, lambda value: print(value == 23))


The first prints 23; the second prints True. The continuation is not necessarily
a printer: it may start another computation or test a result.

These nested continuations are the remaining work at successive call sites.
Part 2 applies this same construction systematically to other control structures.


#### A complete expression: the hypotenuse

For the hypotenuse of a right triangle, compute the two squares, add them,
then take the square root. Each intermediate result becomes the argument of
the next continuation.


In [ ]:
def square_cps(x, k):
    return k(x * x)

def sum_cps(x, y, k):
    return k(x + y)

def root_cps(x, k):
    return k(x ** 0.5)

def hypotenuse_cps(x, y, k):
    return square_cps(x, lambda x2:
        square_cps(y, lambda y2:
            sum_cps(x2, y2, lambda total:
                root_cps(total, k))))

hypotenuse_cps(6, 8, show)


Trace the values: 36, then 64, then 100, then 10. The closure waiting for y2
retains x2, so the two squares can be added. This is why closures are useful
here: each stage carries exactly the values needed by the work that follows.


### Part 2: Translate control structures into CPS

CPS is not restricted to arithmetic examples. We can translate a program by
replacing the caller's implicit remaining work with an explicit continuation.
The rules below cover values, calls, sequencing, conditionals, loops, recursion,
and a success/error model of exceptions. They are not a complete Python
interpreter: features such as finally and generator suspension need additional
rules.


#### A general scheme for translating to CPS

We sketch simple *translation rules* that you can apply mechanically.

We restrict to expressions and functions; this already covers a large
class of programs.


##### constants and variables

Direct style:


```python
42
x
```


CPS form, given a continuation `k`:


```python
k(42)
k(x)
```


We always *call* the continuation with the value.


##### binary operations

Direct style:


```python
x + y
```


CPS form, assuming `x` and `y` are already values:


```python
k(x + y)
```


If `x` and `y` themselves come from computations, we CPS-transform those
computations and chain them using nested continuations.


##### function definition and call

Direct style:


```python
def f(x):
    return e

result = f(10)
```


CPS form:

1. Translate the body of `f`:

   - direct: `return e`
   - CPS:    `CPS(e, k)` i.e. "evaluate e then call k on the result"

2. The CPS version of `f` takes an extra continuation argument:


```python
def example_f_cps(x, k):
    # CPS(e, k)
    ...
```


3. A call `f(10)` in direct style becomes `example_f_cps(10, k)` in CPS.


In [ ]:
def example_f_direct(x):
    return (x + 1) * 2

def example_f_cps(x, k):
    # mechanically: compute (x+1)*2 then call k
    k((x + 1) * 2)


def example_direct():
    return example_f_direct(10)

def example_cps(k):
    # translate "return example_f_direct(10)" as:
    example_f_cps(10, k)

print("example_direct:", example_direct())
example_cps(lambda v: print("example_cps:", v))


Note: We did *not* change the arithmetic; we only replaced `return` with
"call the continuation". In more complex programs, we systematically introduce
continuations at each call site.


#### Conditionals: choose a continuation

An if–then–else selects one of two paths. Represent each path by a continuation
that includes everything to do along that path:


In [ ]:
def choose_cps(condition, then_k, else_k):
    if condition:
        return then_k()
    return else_k()


The condition is already a Boolean value. Only the selected continuation runs.
These continuations take no arguments: their closures capture the values they need.


In [ ]:
def magnitude_direct(x):
    if x >= 0:
        return x
    else:
        return -x

def magnitude_cps(x, k):
    return choose_cps(x >= 0,
                      lambda: k(x),
                      lambda: k(-x))

magnitude_cps(-3, show)


Both branch closures capture k, the common work after finding the magnitude.
There is no need to pass it separately to choose_cps. For example:


In [ ]:
magnitude_cps(-3, lambda value: value + 10)


The result is 13. The negative branch delivers 3 to the continuation, which
adds 10. The positive branch is never called.

Here `return k(...)` forwards the recipient's Python return value. The computed
answer still reaches its destination through k. In Part 1 we ignored that
return value because show only printed; both forms deliver the same answer.


#### Loops: express repetition with recursion

A loop can be represented by recursion, carrying its changing state in function
arguments. We can then apply the same CPS transformation used for other calls.
Converting a loop to recursion is not, by itself, a CPS transformation.

Start with a loop that sums a sequence of integers:


In [ ]:
def total_loop(xs):
    result = 0
    for value in xs:
        result += value
    return result


An equivalent recursive sum adds the current element to the sum of the remaining
elements. Here i is the index of the next element to include:


In [ ]:
def total(xs, i):
    if i == len(xs):
        return 0
    return xs[i] + total(xs, i + 1)

total_loop([3, -2, 5]), total([3, -2, 5], 0)


Both results are 6. After the recursive call returns, we still need to add xs[i].
Make that remaining work a continuation:


In [ ]:
def total_cps(xs, i, k):
    if i == len(xs):
        return k(0)
    return total_cps(
        xs, i + 1,
        lambda subtotal: k(xs[i] + subtotal)
    )

total_cps([3, -2, 5], 0, lambda result: result)


Each lambda captures its own i and k, along with xs. When the base case delivers
0, the saved continuations add 5, then -2, then 3, producing 6. Finally, the
original k receives the answer. For example, replacing it with
`lambda result: result * 10` produces 60 without changing the summation.

Recursion expresses the repetition; CPS makes the remaining computation explicit.
These versions still use Python's call stack. Later we will use a trampoline
to avoid the recursion-depth limit.


#### Recursion: retain the work after the recursive call

For factorial, the remaining work after factorial(n-1) is to multiply by n.
The transformation is the same as for the recursive sum: capture that pending
operation in a continuation.


In [ ]:
def factorial_direct(n):
    if n == 0:
        return 1
    return n * factorial_direct(n - 1)

def factorial_direct_cps(n, k):
    if n == 0:
        return k(1)
    return factorial_direct_cps(n - 1, lambda value: k(n * value))

factorial_direct(3), factorial_direct_cps(3, lambda result: result)


Each new continuation captures its own n and the previous k. At the base case,
the result passes through the saved multiplications: `1 -> 1 -> 2 -> 6`.
The control that the ordinary call stack remembered is now also explicit in
the closure chain.


#### Exceptions: choose where failure goes

An ordinary call has a normal outcome and may also fail. Give the CPS
computation two continuations: on_return for a result and on_error for an error.
A handler changes the error continuation for the operation it protects.

Here a computation has the interface `operation(on_return, on_error)`.
We define four operations that

- deliver a value

- deliver an error

- sequence two computations

- try a computation and, on failure, run a recovery computation


In [ ]:
def return_(x):
  return lambda on_return, on_error: on_return(x)

def raise_(x):
  return lambda on_return, on_error: on_error(x)

def bind(op1, op2):
  """ executes op1 then op2, outputs error if either fails """
  return lambda on_return, on_error: op1(lambda res: op2(res)(on_return, on_error), on_error)

def try_(op1, op2):
  """ executes op1, and if op1 fails, executes op2 """
  return lambda on_return, on_error: op1(on_return, lambda res: op2(res)(on_return, on_error))


In bind, op1 is a computation; op2 is a function that uses its successful result
to construct the next computation. The same error continuation is passed through,
so failure bypasses the remaining normal work.

In try_, the replacement error continuation constructs the recovery computation.
That recovery uses the outer error continuation: a failure inside the handler
is not caught again by the same handler.

For demonstration, our final recipients print the result or error:


In [ ]:
on_return = lambda x: print('ok: ', x)
on_error = lambda x: print('error: ', x)

run_cps = lambda op: op(on_return, on_error)


and a function that might produce an error, depending on its parameters,


In [ ]:
def safe_div(x,y):
  def f(on_return, on_error):
    if y==0:
      return on_error('div by zero')
    else:
      return on_return(x/y)
  return f


In the next example, we apply two safe division commands in sequence, first divide $x/2$ then divide the result by $y$,


In [ ]:
op = lambda x,y: bind(safe_div(x,2),
                      lambda a: safe_div(a,y))
run_cps(op(30,2))
run_cps(op(30,0))


A handler can recover: try to divide $x/y$, and on failure divide $x/10$.
The direct-style idea is:


```python
try:
    answer = x / y
except ZeroDivisionError:
    answer = x / 10
```


In the CPS version, try_ supplies the replacement error continuation:


In [ ]:
op = lambda x,y: try_(safe_div(x,y),
                      lambda _: safe_div(x,10))

run_cps(op(30,0))


The next example propagates the error to the outer recipient. It models re-raising;
it does not execute Python's raise statement:


In [ ]:
op = lambda x,y: try_(safe_div(x,y),
                      lambda msg: raise_(msg))

run_cps(op(30,2))
run_cps(op(30,0))


#### Executing CPS in Python: thunks and trampolines

A continuation receives a result. A *thunk* takes no arguments and represents
a step that has not run yet. Instead of immediately calling the next step,
we can return a thunk and let a loop call it.


In [ ]:
end_cont = lambda value: value


Let's implement a function $f(x,y)$ that computes $2x+y$ in CPS-style.

For that, we create a sum and a multiplication function, that are called in sequence by `f`,


In [ ]:
def add(x, y, cont=end_cont):
  return lambda: cont(x+y)

def mul(x, y, cont=end_cont):
  return lambda: cont(x*y)

# compute 2*x + y
def f(x, y, cont=end_cont):
  cont2 = lambda new_val: add(new_val, y, cont)
  return mul(2, x, cont2)


If we try to evaluate `f` for some input,


In [ ]:
f(3, 10)


it returns a thunk: a function of no arguments that represents the next step.
Calling that thunk performs the multiplication and passes 6 to cont2. It returns
the addition thunk. Calling that second thunk adds 10 and passes 16 to end_cont:


In [ ]:
f(3, 10)()()


A trampoline repeatedly calls each returned thunk until it gets a final value:


In [ ]:
def trampoline(f, *args):
  """ iteratively invokes thunk-returning functions """
  v = f(*args)
  while callable(v):
    v = v()
  return v


Returning a thunk lets the current Python call finish before the next step
starts. In this small implementation, a callable always means another step;
the final answer must therefore be a non-callable value.

To compute the final result using a trampoline:


In [ ]:
trampoline(f, 3, 10)


##### Tail calls

Tail-recursive functions are functions where the recursive call is the last command to be executed. These functions can be optimized to prevent the increasing size of the call stack, since the last call can be replaced by the next recursive call. These process is called tail-call optimization (TCO) which is done automatically by some programming languages.

Let's introduce a helper function that returns the current depth of the call stack. This will be useful to check how the call stack is being used by our recursive functions.


In [ ]:
import traceback

def factory_current_depth():
  """ eval the current call stack depth """
  current_depth = len(traceback.extract_stack())
  return lambda: len(traceback.extract_stack())-current_depth

depth = factory_current_depth()
show_depth = lambda *state: print(f"{depth()*'*'} : {state}")


Let's test it using a standard tail-recursive implementation of the factorial function,


In [ ]:
def fact_tail(n, fac=1):
  show_depth(n)
  if n==0:
    return fac
  return fact_tail(n-1, n*fac)

print(fact_tail(8))


We can observe how the call stack is increasing for each recursive call. Python, by [design](https://stackoverflow.com/questions/13591970), does not do tail-call optimization, unlike most functional programming languages.

For factorial, the work after computing factorial(n-1) is to multiply its answer
by n and deliver that product to cont. Thus `lambda fac: cont(n*fac)` describes
the pending work. To defer that work too, it becomes
`lambda fac: lambda: cont(n*fac)`.

For n=3, descent saves the multiplications by 3, 2 and 1. The base case supplies
1, and those saved continuations produce 1, then 2, then 6. Both descent and
the saved work must take steps through the trampoline:


In [ ]:
def fact_cps(n, cont=end_cont):
  show_depth(n)
  if n == 0:
    return lambda: cont(1)
  return lambda: fact_cps(n-1, lambda fac: lambda: cont(n*fac))

print(trampoline(fact_cps, 8))


Both the recursive descent and the continuation chain return thunks (functions
of no arguments). The trampoline calls each thunk after the previous call has
returned, so neither phase builds a growing Python call stack. Pending work
still occupies memory in closures; we have not made Python optimize tail calls.

In this case, the implementation can be simplified to a sequence of thunk-returning functions, removing the continuation context,


In [ ]:
def fact_tco(n, fac=1):
  show_depth(n)
  if n == 0:
    return fac
  return lambda: fact_tco(n-1, n*fac)

print(trampoline(fact_tco, 8))


##### Check the distinction

- `cont(value)` performs the continuation now.
- `lambda: cont(value)` packages that call for later.
- The trampoline runs one packaged step at a time.

In `fact_cps`, the closure chain remembers the pending multiplications.
In `fact_tco`, the accumulator already contains their product. Both avoid a
growing Python call stack under the trampoline, but they store pending work
differently.

Why would returning a thunk for recursive descent alone be insufficient?
The chain of pending continuations must also defer its calls; otherwise it
can grow the stack when the base case is reached.


### Part 3: Explore the alternatives in one execution

A normal function returns one answer. A search may find none, one, or many.
We can express this by calling its continuation for each successful branch.
A failed branch produces no result.

Our goal is to place n queens on an n-by-n chessboard without two queens
sharing a row, column, or diagonal. We will separate *choosing a position*,
*combining choices*, and *rejecting conflicts*, then assemble the solver.


#### First, deliver every candidate

The idea is:

- if a search does not produce a result, do not call the continuation

- if a search produces several results, call the continuation for each result

- pass the remaining work into each branch

This first example executes a continuation for each element of the given list:


In [ ]:
def member(xs, cont):
  if xs:
    x, *xs = xs
    cont(x)
    return member(xs, cont)


Let's choose a print as the continuation:


In [ ]:
cont_show = lambda *x: print(*x, end='; ')
member([1,2,3], cont_show)


Searches can be combined:


In [ ]:
def search2(xs, ys):
  member(xs,
         lambda x: member(ys,
                          lambda y: cont_show(x,y)))

search2([1,2,3], 'abcd')


And we can apply filters that cut the search if some predicate is not satisfied:


In [ ]:
cont_nothing = lambda x: None # do nothing continuation

def filter_callback(p, cont):
  return lambda *x: cont(*x) if p(x[0]) else cont_nothing(x)

def search3(xs, ys):
  member(xs,
         lambda x: member(ys,
                          lambda y: filter_callback(lambda x: x%2==0, cont_show)(x,y)))

search3([1,2,3], 'abcd')


The call to member does not stop after the first candidate. After its
continuation finishes, it visits the next candidate. In search2, each x
starts a search through all y values. One call explores every pair.
Filtering simply declines to continue a rejected branch.

An ordinary if chooses one branch for its current input. We have not changed
that meaning: we explicitly visit alternatives, and each visit can make its
own decisions.


#### Package the recurring search operations

The examples above deliver answers to a printer. We now collect them as
lists, so one search can be combined with another without printing.

A state is a list of choices made so far. A continuation receives a state and
returns a list of completed answers. A *search generator* transforms a
continuation into a function of state. Here “generator” is this mathematical
role, not a Python function containing `yield`.

The nested functions below use last week's currying pattern. Calling
`literal(value)` fixes a choice; supplying `cont` specifies what should follow
that choice; supplying `state` starts the step. The closures retain the earlier
arguments until they are needed.

Let's consider the generator `literal` that, for a given value, prepends that value to a (partial) state:


In [ ]:
literal = lambda value: lambda cont: lambda state: cont([value]+state)


To observe states, let's add

- `ret` a continuation that just returns the given state as a list

- `run` a function that executes a generator and returns the list of all list of states


In [ ]:
ret = lambda x: [x]

run = lambda gen: gen(ret)([])


Let's try them with `literal`,


In [ ]:
run(literal(1))


The result is `[1](1)`, not `[1]`. The inner list is a state; the outer list
collects answers. An empty outer list means no answers. In particular,
`[[]]` means one successful answer whose state is empty.

Generator `pipe` is used to create a generator that is a sequence of two generators,


In [ ]:
def pipe(gen1, gen2):
  return lambda cont: lambda state: gen1(gen2(cont))(state)


In [ ]:
g1 = literal(1)
g2 = literal(2)
g3 = pipe(g1, g2)

run(g3)


The state travels as `[] -> [1] -> [2, 1]`. The result is `[2, 1](2, 1)` because
literal prepends each choice. In `gen1(gen2(cont))(state)`, gen2(cont)
constructs the continuation for gen1; it does not yet process a state.
Execution starts with gen1, then that continuation runs gen2.

The next two generators will be useful for backtracking purposes in defining a successful branch and a failed branch,


In [ ]:
success = lambda cont: lambda state: cont(state)
fail    = lambda cont: lambda state: []


Generator `do` pipes a given generator a given number of times,


In [ ]:
def do(gen, n):
  if n==0:
    return success
  return pipe(do(gen, n-1), gen)


In [ ]:
g5 = do(g3, 5)
run(g5)


Generator `either` is the one introducing non-determinism into the computation. It receives a list of generators and produces a generator which behaves as a non-determinist choice between all given generators.

The continuation is passed to all generators, in depth-first search style.
There is no random choice here: either explores each supplied alternative
and concatenates its answers.


In [ ]:
def either(*gens):
  return lambda cont: lambda state: [result
                                    for gen in gens
                                    for result in gen(cont)(state)]


One use case of `either`: generator `choice` prepends values $n$ to $1$ to all given states,


In [ ]:
def choice(n):
  if n==0:
    return fail
  return either(literal(n), choice(n-1))


To produce the cartesian product of values $1,2,3$


In [ ]:
g6 = do(choice(3), 3)
run(g6)


The next generator filters states by predicate:


In [ ]:
def filt(p):
  return lambda cont: lambda state: cont(state) if p(state) else []


The next use case filters the previous cartesian product by selecting only those states starting with value $1$,


In [ ]:
first_one_only = lambda st: st[0]==1

run(pipe(g6, filt(first_one_only)))


#### The n-Queen problem

Let's check how to use these combinators to solve the famous n-queen problem.

A board state stores one row number per placed queen. Columns are implicit:
each new placement advances one column and prepends its row. Thus state[0]
is the newest queen, and entry i is i columns away.

For example, `[2, 4]` is safe: the queens are one column apart and two rows
apart. `[2, 3]` conflicts diagonally: both distances are one. Earlier queens
have already been checked, so only the newest queen needs checking against them.

The following function rejects conflicting partial boards:


In [ ]:
def is_valid(state):
  """Return whether the newest queen avoids conflicts with earlier queens."""
  for i in range(1,len(state)):
    if (state[0] == state[i] or       # if same line
        abs(state[0]-state[i]) == i): # or same diagonal
      return False
  return True


With all these tools, backtracking the n-queen problem can be solved by applying the following combinators:


In [ ]:
def queens(n):
  place_a_queen = pipe(choice(n), filt(is_valid))
  return run(do(place_a_queen, n))

print(*queens(6), sep='\n')


#### Read the solver as a search

`choice(n)` tries each row. `filt(is_valid)` rejects a conflict immediately.
`pipe` connects those steps, and `do(..., n)` repeats them for n columns.
Each new row is prepended to the state, so `state[0]` is the newest queen;
the distance i to an earlier entry is also the distance between their columns.

The search explores alternatives sequentially, not concurrently. When one
branch fails, it contributes an empty list; the remaining branches can still
produce answers. Unlike Part 2, these combinators use direct function calls.
They illustrate search structure, not a stack-safe engine for arbitrary depth.


In [ ]:
queens(4)


There are two solutions for four queens. Explain why changing `either` to keep
only the first branch would lose valid solutions, and why checking conflicts
after each placement avoids work that a generate-all-boards approach performs.


### Recap

The central idea is that *control can be explicit*.

1. A continuation represents the work after a result becomes available.
   A closure retains the values that this work needs.
2. Direct-style calls, sequencing, branches, iteration, recursion, and exception
   handling can be expressed by passing the appropriate continuations.
3. Thunks and a trampoline separate describing a next step from executing it.
4. A search can continue after a branch fails or succeeds. One execution can
   visit every alternative in our finite search and collect all valid solutions;
   branches known to be invalid can be pruned.

The striking expression style is visible in the complete search:


```python
run(do(pipe(choice(n), filt(is_valid)), n))
```


Read it as: choose a row, reject conflicts, repeat for n columns, collect the
answers. This is *compositional, expression-oriented programming*. The expression
is compact because we have defined the control behavior of its parts, not
because the search work has disappeared.

An ordinary if commits to one branch for the current state. Our search explicitly
constructs alternatives and carries the remaining computation into each one.
The same program execution can therefore discover all solutions, rather than
stop at the first path it follows.

Last week, closures let functions retain data. Here they retain what to do next.
Control is no longer only implicit in the placement of statements: it is
something our program can represent and compose.


### Optional extensions

These examples extend the three lectures; they are not a fourth required part.
Choose according to time and interest:

- After Part 2: the hypotenuse, Fibonacci and mergesort examples give more
  practice with composition and recursion.
- After Part 3: permutations, tree paths and Pythagorean triples offer other
  search problems; streams reconnect continuations to earlier course material.
- The logic checker is a longer bridge to the later logic-programming lecture.

Run the three core parts first. The optional stream helpers have distinct
names so they do not replace the search combinators used by `queens`.


#### After Part 2: Composing several steps

Let's try a more complex example, computing the hypotenuse of a right triangle:


In [ ]:
def sqrt(x, cont=end_cont):
  return lambda: cont(x**0.5)

def pow2(x, cont=end_cont):
  return lambda: cont(x**2)

def pyth(x, y, cont=end_cont):
  return pow2(x,
              lambda x2: pow2(y,
                              lambda y2: add(x2, y2,
                                             lambda a2b2: sqrt(a2b2, cont)
                                            )
                             )
             )


In [ ]:
print(pyth(6, 8)()()()())
print(trampoline(pyth, 6, 8))


Alternatively,


In [ ]:
def pyth(x, y, cont=end_cont):
  cont_sqrt = lambda a2b2  : sqrt(a2b2, cont)
  cont_add  = lambda x2, y2: add(x2, y2, cont_sqrt)
  cont_y2   = lambda x2    : pow2(y, lambda y2: cont_add(x2,y2))
  cont_x2   =                pow2(x, lambda x2: cont_y2(x2))
  return cont_x2

print(trampoline(pyth, 6, 8))


#### After Part 2: Fibonacci and mergesort

This simplification is even greater for the Fibonacci sequence, that uses two recursive parameters.

Let's check the traditional tail-recursive definition, the cps solution and the simplified implementation:


In [ ]:
def fib_tail(n, a=1, b=1):
  show_depth(n)
  if n < 2:
    return b
  else:
    return fib_tail(n - 1, b, a+b)

print(fib_tail(8))


In [ ]:
def fib_cps_binary(n, cont=end_cont):
  """ binary recursion, CPS version """
  show_depth(n)
  if n < 2:
    return lambda: cont(1)
  return lambda: fib_cps_binary(n-1, lambda x:
                                 lambda: fib_cps_binary(n-2, lambda y:
                                                        lambda: cont(x+y)))

def fib_cps(n, a=1, b=1, cont=end_cont):
  """ linear recursion, CPS version """
  show_depth(n,a,b)
  if n < 2:
    return lambda: cont(b)
  return lambda: fib_cps(n-1, b, a+b, cont)

print(trampoline(fib_cps, 8))


In [ ]:
def fib_tco(n, a=1, b=1):
  show_depth(n, a, b)
  if n < 2:
    return b
  else:
    return lambda: fib_tco(n-1, b, a+b)

print(trampoline(fib_tco, 8))


The next example computes mergesort with continuations.

Ordinary mergesort has recursion depth proportional to $\log n$, not a total
call count of $\log n$. In this CPS version we also defer continuation calls
so the trampoline controls both splitting and combining.


In [ ]:
def merge(xs, ys):
  result = []
  while xs or ys:
    if not xs or not ys:
       return result + xs + ys
    x, *xtail = xs
    y, *ytail = ys
    if x < y:
      result.append(x)
      xs = xtail
    else:
      result.append(y)
      ys = ytail
  return result

def mergesort_cps(xs, cont=end_cont):
  if len(xs) <= 1:
    return lambda: cont(xs)
  mid = int(len(xs) / 2)
  return lambda: mergesort_cps(xs[:mid],
                       lambda L: lambda: mergesort_cps(xs[mid:],
                                               lambda R: lambda: cont(merge(L, R))))


In [ ]:
trampoline(mergesort_cps, [7,5,3,8,1,2,0,10,3])


For more tail recursion on Python check this [package](https://github.com/0scarB/tail-recursive).


#### After Part 3: Permutations and tree paths

We can use the same continuation pattern to enumerate arrangements and paths.

The next function produces all possible insertions of an element into a list:


In [ ]:
def ins(xs, z, cont):
  if not xs:
    cont([z])
  else:
    cont([z]+xs)
    x, *xs = xs
    ins(xs, z, lambda ys: cont([x]+ys))

ins([1,2,3], 4, cont_show)


The previous function can be used to compute permutations:


In [ ]:
def perm(xs, cont):
  if not xs:
    cont([])
  else:
    x, *xs = xs
    perm(xs, lambda ys: ins(ys, x, cont))

perm([1,2,3], cont_show)


If we'd like to have carry state across the search, instead of just making some side-effect, we need to have a continuation that receives a value and a state, and outputs a new state:


In [ ]:
def member_st(xs, cont, state):
  if not xs:
    return state
  else:
    x, *xs = xs
    new_state = cont(x, state)
    return member_st(xs, cont, new_state)


In [ ]:
# eg, map elements of list
cont_mul10 = lambda value, state: state+[10*value]
member_st([1,2,3], cont_mul10, [])


Let's define a continuation that simply adds the current value to the previous state:


In [ ]:
cont_keep = lambda info, state: state+[info]


And let's refactor `ins` and `perm` to return all searches, instead of printing them:


In [ ]:
def ins(xs, z, cont, state):
  if not xs:
    return cont([z], state)
  else:
    new_state = cont([z]+xs, state)
    x, *xs = xs
    return ins(xs, z, lambda ys, st2: cont([x]+ys, st2), new_state)

ins([1,2,3], 4, cont_keep, [])


In [ ]:
def perm(xs, cont, state):
  if not xs:
    return cont(xs, state)
  else:
    x, *xs = xs
    return perm(xs, lambda ys, st2: ins(ys, x, cont, st2), state)

perm([1,2,3,4], cont_keep, [])


##### Searching Tree Paths

Given a binary tree, find all paths from the root to a leaf.

         10
        /  \
       /    \
      5     15
     * \   *  
    1   7 12


In [ ]:
t =  [10,
      [5,  [ 1, None, None], [ 7, None, None]],
      [15, [12, None, None], None]
     ]


In [ ]:
def paths(node, cont):
  if node is None:
    cont([])
  else:
    value, left, right = node
    cont1 = lambda path: cont([value]+path)
    if left is None and right is None:
      paths(None, cont1)
    else:
      if left : paths(left,  cont1)
      if right: paths(right, cont1)

paths(t, cont_show)


#### After Part 3: Pythagorean triples

Let's produce the Pythagorean triples up to a certain limit:


In [ ]:
def increasing(state):
  return len(state)<2 or state[0]<state[1]

def is_pyth(state):
  a,b,c = state
  return a*a + b*b == c*c

def pyth_triples(limit):
  pick_number = pipe(choice(limit), filt(increasing))
  pick_3 = do(pick_number, 3)
  return run(pipe(pick_3, filt(is_pyth)))

pyth_triples(50)


#### After Part 3: Streams and the prime sieve

[Allison90](http://www.allisons.org/ll/Publications/1990BCJ/) also presents a way to implement streams as continuations. A stream is a function which produces the first value of a sequence and another stream; the latter represents the rest of the sequence.

Herein, there are three concepts:

- Sources produce the first value, and is given a sink to use that value

- Agents receive value and send new modified values

- Sinks consume a value, and is given a source that is used to produce more values.

Their types are:

     source :: sink -> list
     sink :: int -> source -> list
     agent :: sink -> sink

The next implementations are very recursive intensive. In Python it's easy to max the call stack. These functions are more appropriate for languages with automatic tail-call optimization.

Some basic sources and sinks:


In [ ]:
null_src = lambda sink: []

# basic sinks
collect  = lambda n: lambda src: [n]+src(collect)
dev_null = lambda n: lambda src: src(dev_null) # abandon hope all ye who enter here


The first source produces values from $a$ to $b$, a bit like Python's `range`.


In [ ]:
# interval :: (int, int) -> source
def interval(a, b):
  def f(sink):
    if a <= b:
      return sink(a)(interval(a+1, b))
    else:
      return []
  return f


In [ ]:
print(interval(1,10)(collect))
print(interval(1,10)(dev_null))


Function `run_stream` links a source, an agent and a link:


In [ ]:
# run_stream :: source -> agent -> source
# run_stream :: source -> agent -> sink -> list
run_stream = lambda src: lambda agent: lambda sink: src(agent(sink))


The next functions implement agents that return even and odd indexed stream values (streams here are 1-indexed),


In [ ]:
def even(sink):
  return lambda n: lambda src: src(odd(sink))

def odd(sink):
  return lambda n: lambda src: sink(n)(lambda sink2: src(even(sink2)))


In [ ]:
run_stream(interval(1,10))(odd)(collect)


Agent `pipe_stream` creates a sequence of two agents,


In [ ]:
# pipe_stream :: agent -> agent -> agent
def pipe_stream(agent1, agent2):
  return lambda sink: lambda n: lambda src: agent1(agent2(sink))(n)(src)


In [ ]:
run_stream(interval(1,10))(pipe_stream(odd, even))(collect)


`alternate` takes two sources and creates a new one that alternates between their values,


In [ ]:
# alternate :: (source, source) -> source
def alternate(src1, src2):
  return lambda sink: src1(lambda n: lambda src1a:
                                sink(n)(alternate(src2, src1a)))


In [ ]:
s1 = interval(11,20)
s2 = interval(21,30)
print(alternate(s1,s2)(collect))


Funciton `tee` makes two copies of a given source,


In [ ]:
# tee :: source -> (source, source)
def tee(src):
  return (src, src)


In [ ]:
print(alternate(*tee(s1))(collect))


Function `split` is an inverse function of `alternate`,


In [ ]:
# split :: source -> (source, source)
def split(src):
  return run_stream(src)(odd), run_stream(src)(even)


In [ ]:
s1 = interval(11,20)
s2, s3 = split(s1)
print(alternate(s3,s2)(collect))


`filter_stream` filters the received values by a given predicate,


In [ ]:
# filter_stream :: (int -> bool) -> agent
def filter_stream(p):
  def f(sink):
    def g(n):
      def h(src):
        if p(n):
          return sink(n)(run_stream(src)(filter_stream(p)))
        else:
          return run_stream(src)(filter_stream(p))(sink)
      return h
    return g
  return f


In [ ]:
run_stream(interval(1,10))(filter_stream(lambda n: n%2==0))(collect)


As a typical example in stream-land, let's source the ancient Sieve of Eratosthenes,


In [ ]:
# sieve :: agent
def sieve(sink):
  return lambda n: lambda src: sink(n)(run_stream(src)(pipe_stream(filter_stream(lambda m: m%n!=0), sieve)))


In [ ]:
run_stream(interval(2,60))(sieve)(collect)


Some ending words from the article:

Continuations allow the programmer to specify the flow of control of a program (this is why they are useful in denotational semantics). They allow backtracking programs such as the generators to be written. They also allow the stream processing functions - sources, sinks and agents - to pass control between each other as coroutines or as processes. At first sight functional languages seem to be poor in control mechanisms, possessing only if, application, composition and recursion. In reality this simplicity is richness and continuations are one way of getting a desired control regime.

Generators and stream processing are important models in solving certain types of problem. Continuations bring them into pure functional programming without the need for new language mechanisms.

Continuations involve a novel style of programming but the programs that result (n-queens, primes) are often close to their English descriptions. They have something of an imperative flavour - do this and then do that - but they are functional and free of side-effects. The programs can be efficiently implemented by optimisations such as tail-recursion.


#### Looking ahead: A logic checker

One application where continuations are useful is search.

Denys Duchier shows [here](https://www.ps.uni-saarland.de/~duchier/python/continuations.html) that search can be achieved by passing *two* continuations:

- a success continuation, used to progress the search

- a failure continuation, to backtrack to a previous choice

Something like


```python
  def search(state, yes, no):
    if check(state):
      return yes(state, no)
    else:
      return no()
```


To implement searching by a second path if the first failed,


```python
  def search(state, yes, no):
    if check(state):
      return path1.search(state,
                          yes,
                          lambda: path2.search(state, yes, no))
```


The next program, written by Denys Duchier, shows how to use this method to find logical assignments to validate a boolean expression.


In [ ]:
# Copyright (c) Feb 2000, by Denys Duchier, Universitaet des Saarlandes

"""
This module implements a validity checker for propositional
formulae.  Its purpose is to illustrate programming with
continuations to implement a `backtracking' search engine.

A formula is represented by an object which responds to
the following methods:

    self.satisfy(environment,yes,no)
    self.falsify(environment,yes,no)

`environment' is a partial assignment of truth values to propositional
variables. `satisfy' attempts to make the formula true, possibly
by appropriately extending the partial assignment. `no' is the
failure continuation.  It takes no argument, and resumes search
in an alternative branch of an earlier choice point. `yes' is the
success continuation and takes 2 arguments: the current partial
assignment environment, and the current failure continuation.
"""


The superclass is `Formula` that includes method `isValid` that tries to falsify an expression. If it can't, then the expression is a tautology.


In [ ]:
class Formula:
  def isValid(self):
    """a formula is valid iff it cannot be falsified"""
    return self.falsify({},
                        lambda environment,no: False,
                        lambda               : True)

  _tracing = id # print

  # satisfy and falsify are wrappers that allow tracing
  # _satisfy and _falsify do the actual work
  def satisfy(self, environment, yes, no):
    Formula._tracing(f'satisfy {self} environment:{environment}')
    return self._satisfy(environment, yes, no)

  def falsify(self, environment, yes, no):
    Formula._tracing(f'falsify {self} environment:{environment}')
    return self._falsify(environment, yes, no)


The next subclasses implement how to satisfy or falsify the respective logical operator:


In [ ]:
class Negation(Formula):
  def __init__(self, p):
    self.p = p

  def __str__(self):
    return f'¬{self.p}'

  def _satisfy(self, environment, yes, no):
    """to satisfy ¬P we must falsify P"""
    return self.p.falsify(environment, yes, no)

  def _falsify(self, environment, yes, no):
    """to falsify ¬P we must satisfy P"""
    return self.p.satisfy(environment, yes, no)


In [ ]:
class Conjunction(Formula):
  def __init__(self, p, q):
    self.p = p
    self.q = q

  def __str__(self):
    return f'({self.p} ∧ {self.q})'

  def _satisfy(self, environment, yes, no):
    """to satisfy P∧Q we must satisfy both P and Q"""
    return self.p.satisfy(
      environment,
      lambda environment,no,self=self,yes=yes:
        self.q.satisfy(environment, yes, no),
      no
    )

  def _falsify(self, environment, yes, no):
    """to falsify P∧Q we can falsify either P or Q"""
    return self.p.falsify(
      environment,
      yes,
      lambda self=self,environment=environment,yes=yes,no=no:
        self.q.falsify(environment, yes, no)
    )


In [ ]:
class Disjunction(Formula):
  def __init__(self,p,q):
    self.p = p
    self.q = q

  def __str__(self):
    return f'({self.p} ∨ {self.q})'

  def _satisfy(self, environment, yes, no):
    """to satisfy P∨Q we can satisfy either P or Q"""
    return self.p.satisfy(
      environment,
      yes,
      lambda self=self,environment=environment,yes=yes,no=no:
        self.q.satisfy(environment, yes, no)
    )

  def _falsify(self, environment, yes, no):
    """to falsify P∨Q we must falsify both P and Q"""
    return self.p.falsify(
      environment,
      lambda environment,no,self=self,yes=yes:
        self.q.falsify(environment, yes, no),
      no
    )


Class `Variable` is where the environment is populated with assignments


In [ ]:
class Variable(Formula):
  def __init__(self,v):
    self.v = v

  def __str__(self):
    return self.v

  def bind(self, value, environment):
    """returns a new partial assignment that additionally
       assigns the truth 'value' to this propositional variable"""
    environment = environment.copy()
    environment[self.v] = value
    return environment

  def assign(self, value, environment, yes, no):
    """attempts to assign the given truth value to this proposition
       variable.  If environment already contains a contradictory
       assignment, the failure continuation is invoked. Otherwise, environment
       is extended if necessary and the success continuation is invoked."""
    if self.v in environment:
      return yes(environment, no) if environment[self.v]==value else no()
    else:
      return yes(self.bind(value, environment), no)

  def _satisfy(self, environment, yes, no):
    """to satisfy a propositional variable, we must assign it true"""
    return self.assign(True, environment, yes, no)

  def _falsify(self, environment, yes, no):
    """to falsify a propositional variable, we must assign it false"""
    return self.assign(False, environment, yes, no)


The next auxiliary functions are useful to facilitate the writing of logical formulas:


In [ ]:
def AND(*args):
  """n-ary version of Conjunction"""
  formula, *args = args
  for x in args:
    formula = Conjunction(formula,x)
  return formula

def OR(*args):
  """n-ary version of Disjunction"""
  formula, *args = args
  for x in args:
    formula = Disjunction(formula,x)
  return formula

def NOT(x):
  return Negation(x)

def IF(p,q):
  return OR(NOT(p),q)

def IFF(p,q):
  return AND(IF(p,q), IF(q,p))

def XOR(p,q):
  return OR(AND(p,NOT(q)), AND(NOT(p),q))


A use case:


In [ ]:
P = Variable('P')
Q = Variable('Q')
R = Variable('R')

# is [(P∨Q) ∧ (P⇒R) ∧ (Q⇒R)] ⇒ R a tautology?
form = IF(AND(OR(P,Q),
              IF(P,R),
              IF(Q,R)),
          R)

Formula._tracing = print # hack to see what's happening under the hood
form.isValid()


## References

The CPS translation material and examples are adapted from the earlier course
notes. The search-combinator construction follows L. Allison's work on
continuations, generators, and streams.

- [L. Allison: Continuations, Generators, and Streams](http://www.allisons.org/ll/Publications/1990BCJ/)
- [Eli Bendersky: On Recursion, Continuations and Trampolines](https://eli.thegreenplace.net/2017/on-recursion-continuations-and-trampolines/)
